In [ ]:
import csv
import random

def generate_flight(filename="flight_log.csv"):
    # Fija semilla opcional para reproducibilidad
    random.seed(42)

    dt = 0.1          # Paso de tiempo: 100 ms (10 Hz)
    g = 9.81          # Gravedad (m/s^2)
    T0 = 20.0         # Temperatura inicial en rampa (°C)
    lapse_rate = 0.0065  # Gradiente térmico estándar (°C/m)

    # Variables de estado inicial
    t = 0.0
    h = 0.0
    v = 0.0
    rows = []

    def record_sample(time_s, alt_true, accel_net):
        # 1. Ruido gaussiano en barómetro (sigma = 0.5 m)
        alt_noisy = alt_true + random.gauss(0.0, 0.5)

        # 2. Lectura real del acelerómetro (fuerza específica + pequeño ruido de sensor)
        accel_sensor = (accel_net + g) + random.gauss(0.0, 0.05)

        # 3. Temperatura según altitud real (+ pequeño ruido de termistor)
        temp_c = (T0 - lapse_rate * max(0.0, alt_true)) + random.gauss(0.0, 0.05)

        rows.append([
            int(round(time_s * 1000)),
            round(alt_noisy, 3),
            round(accel_sensor, 3),
            round(temp_c, 2)
        ])

    # --- FASE 1: En rampa (2.0 s) ---
    steps_ramp = int(2.0 / dt)
    for _ in range(steps_ramp):
        record_sample(t, h, 0.0)
        t += dt

    # --- FASE 2: Propulsión (1.5 s, aceleración neta = +60 m/s^2) ---
    steps_boost = int(1.5 / dt)
    a_boost = 60.0
    for _ in range(steps_boost):
        record_sample(t, h, a_boost)
        h += v * dt + 0.5 * a_boost * (dt ** 2)
        v += a_boost * dt
        t += dt

    # --- FASE 3: Vuelo balístico (solo gravedad hasta el apogeo, v <= 0) ---
    a_coast = -g
    while v > 0:
        record_sample(t, h, a_coast)
        h += v * dt + 0.5 * a_coast * (dt ** 2)
        v += a_coast * dt
        t += dt

    print(f"Apogeo real simulado: {h:.2f} m en t = {t:.1f} s")

    # --- FASE 4: Paracaídas (descenso a velocidad casi constante ~ -6 m/s) ---
    v = -6.0
    while h > 0:
        record_sample(t, h, 0.0)
        h += v * dt
        if h < 0:
            h = 0.0
        t += dt

    # --- FASE 5: En tierra (3.0 s) ---
    steps_landed = int(3.0 / dt)
    for _ in range(steps_landed):
        record_sample(t, 0.0, 0.0)
        t += dt

    # --- INSERCIÓN DE 3 LÍNEAS CORRUPTAS (paquetes perdidos) ---
    # 1. Campo vacío en altitud
    rows[25] = [rows[25][0], "", rows[25][2], rows[25][3]]
    # 2. Texto 'ERR' en aceleración
    rows[60] = [rows[60][0], rows[60][1], "ERR", rows[60][3]]
    # 3. Columna faltante (solo 3 columnas en vez de 4)
    rows[110] = [rows[110][0], rows[110][1], rows[110][2]]

    # Escribir archivo CSV
    with open(filename, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["time_ms", "altitude_m", "accel_z_ms2", "temp_c"])
        writer.writerows(rows)

    print(f"Archivo '{filename}' generado con {len(rows)} registros (incluyendo 3 corruptos).")

if __name__ == "__main__":
    generate_flight()

Apogeo real simulado: 480.34 m en t = 12.7 s
Archivo 'flight_log.csv' generado con 958 registros (incluyendo 3 corruptos).
